In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/consolidated-pipeline/1st_setup/Utilities

In [0]:
print(bronze_schema,silver_schema,gold_schema)

Using widgets for Reusability,UI control

In [0]:
dbutils.widgets.text("Catalog","fmcg","catalog")
dbutils.widgets.text("Data_Source","customers","data_source")

In [0]:
catalog= dbutils.widgets.get("Catalog")
data_source=dbutils.widgets.get("Data_Source")
print(catalog,data_source)

In [0]:
base_path = f"s3://amzn-s3-sportsbar/{data_source}/*.csv"
print(base_path)

In [0]:

df= (spark.read.format("csv")
    .option("header",True)
    .option("inferSchema",True)
    .option("mode","PERMISSIVE")
    .option("multiline","true")
    .load(base_path)
    .withColumn("readTimeStamp",F.current_timestamp())
    .select("*","_metadata.file_size","_metadata.file_name")
    )
    
display(df.limit(10))

In [0]:
df.printSchema()

In [0]:
df.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed","true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

## Silver

In [0]:
df_bronze= spark.sql(f"SELECT * from {catalog}.{bronze_schema}.{data_source}")
display(df_bronze.limit(10))

In [0]:
df_bronze.printSchema()

### # Checking data irregularity in bronze table

In [0]:
df_bronze.createOrReplaceTempView("bronze_table")  
df3 =spark.sql(f"SELECT * from bronze_table  where city IS NULL")
display(df3)

In [0]:
df3 =spark.sql(f"SELECT count(customer_id) as cnt , customer_id as COUNT \
               from bronze_table \
               group by customer_id  \
               having cnt >1")
display(df3)
# df4 = spark.sql(f"SELECT * FROM bronze_table where customer_id='789503'")
# display(df4)

In [0]:
print("Total Rows before dropping duplicate",df_bronze.count())
df_silver=df_bronze.drop_duplicates(["customer_id"])
print("Total Rows after dropping duplicate",df_silver.count())

In [0]:
# checking how many customer_name have initial spacing
display(
    df_silver.filter(F.col("customer_name") !=F.trim(F.col("customer_name")))
)

In [0]:
df_silver=df_silver.withColumn("customer_name",
    F.trim(F.col("customer_name"))
)
display(
    df_silver.filter( F.col("customer_name")!= F.trim(F.col("customer_name")))
)

In [0]:
df_silver.createOrReplaceTempView("silver_table")
temp_df=spark.sql("Select * from silver_table where city IS null")
display(temp_df)

In [0]:
df_silver.select("city").distinct().show()

Fixing valid city names

In [0]:
city_mapping= {
  "Bengaluruu" : "Bengaluru",
  "Bengalore":  "Bengaluru",
  "NewDelhee" : "New Delhi",
  "NewDheli" : "New Delhi",
  "NewDelhi": "New Delhi",
  "Hyderabadd" :"Hyderabad",
  "Hyderbad" : "Hyderabad"
}
allowed= ["Bengaluru","Hyderabad","New Delhi"]
df_silver= (df_silver
  .replace(city_mapping,subset=["city"])
  .withColumn(
  "city",
  F.when(F.col("city").isNull(),None)
  .when(F.col("city").isin(allowed),F.col("city"))
  .otherwise(None)
  )
)

In [0]:
df_silver.select("city").distinct().show()

In [0]:
df_silver.select("customer_name").distinct().show()

Many city names have same name but capital/small differnce, fixing that

In [0]:
df_silver= (df_silver.withColumn("customer_name",
                       F.when(F.col("customer_name").isNull(),None)
                        .otherwise(F.initcap("customer_name"))
                        ))


In [0]:
df_silver.select("customer_name").distinct().show()

In [0]:
df_silver.filter(F.col("city").isNull()).show()

In [0]:
df_silver.createOrReplaceTempView("silve_table")
tem_df=spark.sql("Select * from silve_table where city IS null")
display(tem_df)

In [0]:
# Let's fix the customer with null city
customer_city_fix= {
    789403 : "New Delhi",
    789420 : "Bengaluru",
    789521 : "Hyderabad",
    789603 : "Hyderabad"
}
df_fix = spark.createDataFrame(
    [(k, v) for k, v in customer_city_fix.items()],
    ["customer_id", "fixed_city"]
)

display(df_fix)

In [0]:
# Joining both df
df_silver =(
    df_silver.join(df_fix, "customer_id", "left")
    .withColumn("city", F.coalesce("city","fixed_city"))
).drop("fixed_city")

In [0]:
# Verifying
df_silver.filter(F.col("customer_id")==789420).show()

In [0]:
df_silver.show()

In [0]:
df_silver.printSchema()


In [0]:
df_silver = df_silver.withColumn("customer_id",F.col("customer_id").cast("string"))
df_silver.printSchema()

In [0]:
# making Chnages according to golden table
df_silver = (df_silver
.withColumn("customer", F.concat_ws("-","customer_name",F.coalesce(F.col("city"),F.lit("unknown"))))
.withColumn("market",F.lit("India"))
.withColumn("plateform",F.lit("Sports-Bar"))
.withColumn("channel",F.lit("Acquitions")))

In [0]:
display(df_silver)

In [0]:
df_silver.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed","true")\
    .option("mergeSchema","true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")


## Gold Processing


In [0]:
df_silver_processed_data = spark.sql(f"Select * From {catalog}.{silver_schema}.{data_source};")

In [0]:
df_gold = df_silver_processed_data.select("customer_id","customer","customer_name","city" ,"market","plateform", "channel")
display(df_gold)

In [0]:
df_gold = df_gold.withColumnRenamed("customer_id","customer_code")

In [0]:
display(df_gold)

Write this data to gold catalog


In [0]:
df_gold.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed","true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")

In [0]:
print(catalog, gold_schema, data_source)

##Merging both gold tables


In [0]:
delta_table=DeltaTable.forName(spark,"fmcg.gold.dim_customers")
df_child_customer = spark.table("fmcg.gold.sb_dim_customers")\
.select("customer_code","customer","market","plateform","channel").withColumnRenamed("plateform","platform")

In [0]:
display(df_child_customer)

In [0]:
# upsert operation
delta_table.alias("target")\
    .merge(
        source= df_child_customer.alias("source"),
        condition = "target.customer_code = source.customer_code"
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()